# BOCSAR Crime Data Cleaning

## Overview
This notebook cleans and processes suburb-level crime data from BOCSAR (Bureau of Crime Statistics and Research NSW) for use in the Sydney Liveability AI project.

## Data Source
- **File:** `SuburbData25Q4.csv`
- **Downloaded from:** [BOCSAR Open Datasets](https://bocsar.nsw.gov.au/statistics-dashboards/open-datasets.html)
- **Type:** Recorded Criminal Incidents by month — by Suburb

## What This Notebook Does
1. Loads the raw suburb-level crime CSV
2. Filters for the 5 MVP suburbs: Newtown, Glebe, Redfern, Surry Hills, Haymarket
3. Selects the most recent 24 months of data (Jan 2024 – Dec 2025)
4. Calculates `annual_avg` — the monthly average incident count per offence category
5. Exports the cleaned data to `data/processed/bocsar_clean.csv`

## Output
`data/processed/bocsar_clean.csv` with columns:
- `suburb` — suburb name
- `sa4_area` — N/A (suburb-level data available directly)
- `crime_type` — crime category
- `year` — data period (2024-2025)
- `incident_count` — average monthly incidents over 24 months

## Notes
- Suburb-level data was available from BOCSAR, so SA4-level mapping was not required.
- `sa4_area` is set to N/A as data comes directly from suburb level.
- `incident_count` is calculated from Jan 2024 to Dec 2025 (24 months).

In [ ]:
import pandas as pd
import os

# load csv
base_path = os.path.abspath(os.path.join(os.getcwd(), ".."))
df = pd.read_csv(os.path.join(base_path, "data/raw/bocsar/SuburbData25Q4.csv"))

print("Shape:", df.shape)

col1 = pd.DataFrame(df.columns.tolist(), columns=['Columns'])
col2 = pd.DataFrame(df['Suburb'].unique(), columns=['Unique Suburbs'])

display(pd.concat([col1, col2], axis=1))

Shape: (279496, 375)


,Columns,Unique Suburbs
0,Suburb,Aarons Pass
1,Offence category,Abbotsbury
2,Subcategory,Abbotsford
3,Jan 1995,Abercrombie
4,Feb 1995,Abercrombie River
...,...,...
4503,NaN,Yowrie
4504,NaN,Yullundry
4505,NaN,Yuraygir
4506,NaN,Zara


## EDA Findings

- Dataset contains 279,496 rows across 4,508 NSW suburbs
- Suburb-level data was available directly — SA4 mapping not required
- All 5 MVP suburbs confirmed present in the dataset
- Most recent 24 months: Jan 2024 – Dec 2025
- No missing values found in the filtered subset (310 rows)

### Limitation
Original task assumed only SA4-level data would be available.
Since suburb-level data exists, each MVP suburb has its own
independent crime figures — Newtown and Glebe are NOT sharing data.
This is an improvement over the original SA4 mapping plan.

In [13]:
# Five suburb filters
suburbs = ['Newtown', 'Glebe', 'Redfern', 'Surry Hills', 'Haymarket']
df_filtered = df[df['Suburb'].isin(suburbs)]

print("Shape after filter:", df_filtered.shape)
print("Suburbs found:", df_filtered['Suburb'].unique())

In [14]:
# Select the most recent 24 months
month_cols = [col for col in df_filtered.columns if col not in ['Suburb', 'Offence category', 'Subcategory']]

# Convert column names to datetime for sorting
month_dates = pd.to_datetime(month_cols, format='%b %Y')

# Get the most recent 24 months
recent_24 = [month_cols[i] for i, d in enumerate(month_dates) if d >= month_dates[-24]]

pd.DataFrame(recent_24, columns=['Month'])

,Month
0,Jan 2024
1,Feb 2024
2,Mar 2024
3,Apr 2024
4,May 2024
5,Jun 2024
6,Jul 2024
7,Aug 2024
8,Sep 2024
9,Oct 2024


In [15]:
# Calculate annual average from recent 24 months
df_recent = df_filtered[['Suburb', 'Offence category', 'Subcategory'] + recent_24].copy()

df_recent['annual_avg'] = df_recent[recent_24].mean(axis=1)

print(df_recent[['Suburb', 'Offence category', 'Subcategory', 'annual_avg']].head(10))

In [16]:
# Keep only relevant columns
df_output = df_recent[['Suburb', 'Offence category', 'annual_avg']].copy()

# Rename columns to match ORM model
df_output.columns = ['suburb', 'crime_type', 'incident_count']

# Add required columns
df_output['sa4_area'] = 'N/A'
df_output['year'] = '2024-2025'

# Reorder columns
df_output = df_output[['suburb', 'sa4_area', 'crime_type', 'year', 'incident_count']]

# Save to processed folder
output_path = os.path.join(base_path, "data/processed/bocsar_clean.csv")
df_output.to_csv(output_path, index=False)

print("Saved to:", output_path)
print("Shape:", df_output.shape)
print(df_output.head())

Saved to: /Users/amanda/Desktop/sydney-liveability-ai/data/processed/bocsar_clean.csv
Shape: (310, 5)
       suburb sa4_area crime_type       year  incident_count
101432  Glebe      N/A   Homicide  2024-2025        0.000000
101433  Glebe      N/A   Homicide  2024-2025        0.000000
101434  Glebe      N/A   Homicide  2024-2025        0.000000
101435  Glebe      N/A   Homicide  2024-2025        0.041667
101436  Glebe      N/A    Assault  2024-2025        7.541667
